In [1]:
%load_ext dotenv
%dotenv

In [ ]:
from qdrant_client import QdrantClient
#from qdrant_client.models import VectorParams, Distance
from llama_index.core import VectorStoreIndex, Settings, Document, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from datasets import load_dataset
import os
import logging

logging.basicConfig(level=logging.INFO)

/Users/sim/repo/chainlit-samples/.venv/lib/python3.12/site-packages/qdrant_client/http/models/models.py:758: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is empty, alternative syntax for `is_empty: \&quot;field_name\&quot;`",
/Users/sim/repo/chainlit-samples/.venv/lib/python3.12/site-packages/qdrant_client/http/models/models.py:762: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is null, alternative syntax for `is_null: \&quot;field_name\&quot;`",


In [3]:
client = QdrantClient(path=os.getenv('QDRANT_PATH'))

In [ ]:
"""
client = QdrantClient(
    url=f"{os.getenv('QDRANT_HOST')}:{os.getenv('QDRANT_PORT')}",
    api_key=os.getenv('QDRANT_API_KEY'),
    )
"""

In [4]:
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device="mps",
    embed_batch_size=10,
)

Settings.embed_model = embed_model
Settings.chunk_size = 512
Settings.chunk_overlap = 50

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:sentence_transformers.SentenceTransformer:2 prompts are loaded, with the keys: ['query', 'text']


In [5]:
def rebuild_index(client: QdrantClient):
    try:
        news_dataset = load_dataset(
            "RealTimeData/bbc_news_alltime", "2017-12", split="train"
        )
        logging.info(f"Loaded the BBC News dataset with {len(news_dataset)} rows")
        logging.info(f"Successfully loaded the BBC News dataset with {len(news_dataset)} rows.")
    except Exception as e:
        raise ValueError(f"Error loading the BBC News dataset: {str(e)}")

    news_articles = news_dataset["content"]
    unique_articles = set()
    for article in news_articles:
        if article:
            unique_articles.add(article)
    unique_news_articles = list(unique_articles)
    logging.info(f"We have {len(unique_news_articles)} unique articles in our database.")

    articles = [article for article in unique_news_articles if article and len(article) <= 50000]

    documents = [Document(text=t) for t in articles]
    vector_store = QdrantVectorStore(client=client, collection_name=os.getenv('QDRANT_COLLECTION'))
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(documents, storage_context=storage_context, show_progress=True)
    return index

In [9]:
if client.collection_exists(os.getenv('QDRANT_COLLECTION')):
    vector_store = QdrantVectorStore(client=client, collection_name=os.getenv('QDRANT_COLLECTION'))
    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
else:
    index = rebuild_index(client)

retriever = index.as_retriever(
    similarity_top_k=5,
)

In [10]:
print(os.getenv('QDRANT_COLLECTION'))
client.collection_exists(collection_name=os.getenv('QDRANT_COLLECTION'))

sample


True

In [12]:
response = retriever.retrieve("Trading tariff between China and US")
for r in response:
    print("\n----------\n")
    print(r)


----------

Node ID: ca568b65-0109-48f0-bb89-43a69f285820
Text: Parts of Bombardier's C-Series planes are made in Belfast  The
US has ruled that Canada's Bombardier received government subsidies
and sold C-Series jets below cost in the US, a step likely to lead to
steep tariffs.  The US Commerce Department investigated the aerospace
firm's US sales after a petition from rival American company Boeing.
The co...
Score:  0.694


----------

Node ID: 57ac97b6-3c30-4be3-b3a4-4f0f4c86ee8f
Text: As prime minister, David Cameron said the UK and China were in a
"golden era" of trade relations  David Cameron is to take on a new
role leading a UK government-backed investment initiative between
Britain and China.  The former prime minister will take charge of a
£750m ($1bn) fund to improve ports, roads and rail networks between
China and its...
Score:  0.659


----------

Node ID: d410bd06-8399-4a40-b329-c5294f923a6a
Text: So the document emphasises the economy and fair trade as
security issues, 